常數

In [1]:
MODEL = "./models/BlazePose/pose_landmarker_full.task"

##### 忽略警告

In [2]:
import warnings

warnings.filterwarnings("ignore")

##### 主程式

套件

In [3]:
# view result
import cv2
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg

# data process
import numpy as np
import time

# datatype
from mediapipe.tasks.python.components.containers.landmark import NormalizedLandmark
from mpl_toolkits.mplot3d.axes3d import Axes3D

# custom
from src.mediapipe_lib.base import PoseLandmarkerLiveStream, PoseResult, ResultAnalyzer
from src.utils.plot_painter import draw_lines, draw_dots, set_data_range
from src.utils.process_data import translate_multi_kpt_to_plot_pos

啟動攝影機並開始運算

In [4]:
is_3d = True

range_x = [-1, 1]
range_y = [-1, 1]
range_z = [0, 2]

c_kpts = "#000"
c_left = "#f00"
c_center = "#0f0"
c_right = "#00f"

In [5]:
lanmarker_live = PoseLandmarkerLiveStream(MODEL)
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Can't receive frame (stream end?). Exiting ...")
        break

    # 運算模型
    lanmarker_live.detect_async(frame, int(time.time() * 1000))
    # 模型運算結果(會實時更新)
    cur_result = lanmarker_live.result

    # 初始化表格
    fig = plt.figure()
    ax: Axes3D = fig.add_subplot(projection="3d")
    set_data_range(range_x, range_y, range_z, ax)

    # 可取得身體姿勢結果
    if len(cur_result.pose_landmarks) > 0:
        # 建立分析器
        pose_result = PoseResult(cur_result)
        analyzer = ResultAnalyzer(pose_result)

        # 取得關鍵點和線條資料
        kpts = pose_result.get_all_kpt_positions(get_3d=is_3d)
        l_left, l_center, l_right = analyzer.get_line_positions(get_3d=is_3d)

        # 轉換關鍵點和線條資料
        kpts, l_left, l_center, l_right = translate_multi_kpt_to_plot_pos(
            kpts, kpts, l_left, l_center, l_right
        )

        # 繪製表格
        draw_lines(l_left, is_3d, c_left, "left", ax)
        draw_lines(l_center, is_3d, c_center, "center", ax)
        draw_lines(l_right, is_3d, c_right, "right", ax)
        draw_dots(kpts, is_3d, 5, c_kpts, "kpts", ax)
        ax.legend()

    # 表格轉換成圖片
    canvas = FigureCanvasAgg(fig)
    canvas.draw()
    plt.close()

    # 顯示 3D 姿勢結果
    np_fig = np.asarray(canvas.buffer_rgba())
    cv2.imshow("result", np_fig)

    if cv2.waitKey(1) == ord("q"):  # 按 Q 離開
        break

# 關閉物件
cap.release()
cv2.destroyAllWindows()
lanmarker_live.close()

備用

In [6]:
# cap.release()
# cv2.destroyAllWindows()
# lanmarker_live.close()